# Functions 7 & 8 — Simple PyTorch NN Next-Point suggestion

This notebook:
1. Loads all available data for Functions 7 and 8 (initial + weeks)
2. Fits a small PyTorch MLP regressor for each function
3. Suggests the next point by sampling candidate points in [0, 1]


## Step 0: Import Packages


In [ ]:
import numpy as np
from pathlib import Path
import sys
import json

sys.path.append('../..')  # project root

import importlib
        import scripts.utils.data_utils as data_utils
        importlib.reload(data_utils)

        from scripts.utils.data_utils import (
            load_initial_data,
            load_week1_data,
            load_week2_data,
            load_week3_data,
            load_week4_data,
            _load_latest_week_data,
        )

from scripts.neural_networks import train_model, suggest_next_point


## Step 1: Load Data for Functions 7 and 8 (includes Week 4)


In [2]:
base_dir = Path('../../data/initial_data')
initial_data = load_initial_data(base_dir)

def load_function_dataset(function_num: int):
    function_name = f'function_{function_num}'
    inputs = np.array(initial_data[function_name]['inputs'], dtype=np.float32)
    outputs = np.array(initial_data[function_name]['outputs'], dtype=np.float32).reshape(-1)

    weeks = [1, 2, 3, 4]  # include week 4 data
    loaded_weeks = []
    for week_num in weeks:
        week_dir = Path(f'../../data/week{week_num}')
        if not (week_dir / 'inputs.txt').exists():
            continue
        if week_num == 1:
            week_data = load_week1_data(week_dir)
        elif week_num == 2:
            week_data = load_week2_data(week_dir)
        elif week_num == 3:
            week_data = load_week3_data(week_dir)
        else:
            week_data = _load_latest_week_data(week_dir, f'Week {week_num}')

        week_input = np.atleast_2d(week_data[function_name]['input']).astype(np.float32)
        week_output = np.array([week_data[function_name]['output']], dtype=np.float32).reshape(-1)
        inputs = np.vstack([inputs, week_input])
        outputs = np.concatenate([outputs, week_output])
        loaded_weeks.append(week_num)

    return inputs, outputs, loaded_weeks

X7, y7, weeks7 = load_function_dataset(7)
X8, y8, weeks8 = load_function_dataset(8)

print(f'Function 7: {X7.shape[0]} samples, input dim {X7.shape[1]}, weeks loaded: {weeks7}')
print(f'Function 8: {X8.shape[0]} samples, input dim {X8.shape[1]}, weeks loaded: {weeks8}')


Function 7: 34 samples, input dim 6, weeks loaded: [1, 2, 3, 4]
Function 8: 44 samples, input dim 8, weeks loaded: [1, 2, 3, 4]


## Step 2: Train the Network and Suggest Next Points


In [3]:
# Tune these if you want
epochs = 1200
hidden = 64
lr = 1e-3
samples = 50000

# Function 7
model7, ymean7, ystd7 = train_model(X7, y7, hidden=hidden, epochs=epochs, lr=lr)
x7_next, y7_pred = suggest_next_point(model7, X7.shape[1], ymean7, ystd7, samples=samples)

# Function 8
model8, ymean8, ystd8 = train_model(X8, y8, hidden=hidden, epochs=epochs, lr=lr)
x8_next, y8_pred = suggest_next_point(model8, X8.shape[1], ymean8, ystd8, samples=samples)

print('Function 7 next point:')
print('  ', np.round(x7_next, 6))
print(f'  Predicted score: {y7_pred:.6f}')

print('Function 8 next point:')
print('  ', np.round(x8_next, 6))
print(f'  Predicted score: {y8_pred:.6f}')


Function 7 next point:
   [0.010064 0.250273 0.457877 0.238727 0.383079 0.584059]
  Predicted score: 2.603480
Function 8 next point:
   [0.027137 0.168897 0.007204 0.613151 0.986073 0.162751 0.116094 0.624013]
  Predicted score: 9.929558


## Step 3: Save Results to JSON Files

In [8]:
# Create results directory if it doesn't exist
results_dir = Path('results')
results_dir.mkdir(exist_ok=True)

# Function 7 - Save to JSON
function7_result = {
    "function_name": "function_7",
    "next_point": x7_next.tolist(),
    "formatted": '-'.join([f'{x:.6f}' for x in x7_next]),
    "method": "pytorch_neural_network",
    "predicted_score": float(y7_pred),
    "epochs": epochs,
    "hidden": hidden,
    "lr": lr,
    "samples": samples
}

function7_path = results_dir / 'function7_next_point.json'
with open(function7_path, 'w') as f:
    json.dump(function7_result, f, indent=2)
print(f'✓ Function 7 result saved to: {function7_path}')

# Function 8 - Save to JSON
function8_result = {
    "function_name": "function_8",
    "next_point": x8_next.tolist(),
    "formatted": '-'.join([f'{x:.6f}' for x in x8_next]),
    "method": "pytorch_neural_network",
    "predicted_score": float(y8_pred),
    "epochs": epochs,
    "hidden": hidden,
    "lr": lr,
    "samples": samples
}

function8_path = results_dir / 'function8_next_point.json'
with open(function8_path, 'w') as f:
    json.dump(function8_result, f, indent=2)
print(f'✓ Function 8 result saved to: {function8_path}')

print('\n' + '=' * 60)
print('SUMMARY')
print('=' * 60)
print(f'Function 7: {function7_result["formatted"]}')
print(f'  Predicted score: {y7_pred:.6f}')
print(f'Function 8: {function8_result["formatted"]}')
print(f'  Predicted score: {y8_pred:.6f}')
print('=' * 60)

✓ Function 7 result saved to: results/function7_next_point.json
✓ Function 8 result saved to: results/function8_next_point.json

SUMMARY
Function 7: 0.010064-0.250273-0.457877-0.238727-0.383079-0.584059
  Predicted score: 2.603480
Function 8: 0.027137-0.168897-0.007204-0.613151-0.986073-0.162751-0.116094-0.624013
  Predicted score: 9.929558
